## Introduction
Can a CNN still classify images correctly when someone slightly modifies the image

## Importing the Libraries
NumPy is used for numerical computation.

Matplotlib is used for data visualization.

TensorFlow provides access to the CIFAR-10 dataset.

Scikit-learn is used for splitting the dataset into training and validation sets.


In [11]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tensorflow.keras.datasets import cifar10
from sklearn.model_selection import train_test_split
import os
os.makedirs('figures', exist_ok=True)




## CIFAR-10 dataset
CIFAR-10 is an image classification dataset developed by researchers at the Canadian Institute for Advanced Research. It contains 60,000 colour images divided into 10 object categories.

## Load The Dataset

In [2]:
print("\n LOADING CIFAR-10 DATASET")
(X_train, y_train), (X_test, y_test) = cifar10.load_data()
y_train = y_train.flatten()
y_test = y_test.flatten()

class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

print(f" Dataset loaded successfully")
print(f"  Training set: {X_train.shape}")
print(f"  Test set: {X_test.shape}")


 LOADING CIFAR-10 DATASET
 Dataset loaded successfully
  Training set: (50000, 32, 32, 3)
  Test set: (10000, 32, 32, 3)


## Exploring the dataset Structure

Dataset Specifications:
- Total images: 60,000
- Training set: 50,000 images
- Test set: 10,000 images
- Image dimensions: 32×32 pixels (RGB)
- Color channels: 3 (Red, Green, Blue)
- Number of classes: 10
  * Airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck
- Image format: Numpy arrays
- Pixel value range: 0-255 (unsigned 8-bit integer)

Data Source:
https://www.cs.toronto.edu/~kriz/cifar.html

## Data Preprocessing

In [3]:
print("\n PREPROCESSING DATA")
print(f"  Original data type: {X_train.dtype}")
print(f"  Original value range: [{X_train.min()}, {X_train.max()}]")

# Normalize to [0, 1]
X_train = X_train.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0

print(f"  Converted to float32")
print(f"  Normalized to [0, 1] range")
print(f"  New value range: [{X_train.min()}, {X_train.max()}]")

# Train/validation split
X_train_split, X_val, y_train_split, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
)
print(f"    Train/validation split applied")
print(f"    Training: {X_train_split.shape}")
print(f"    Validation: {X_val.shape}")
print(f"    Test: {X_test.shape}")


 PREPROCESSING DATA
  Original data type: uint8
  Original value range: [0, 255]
  Converted to float32
  Normalized to [0, 1] range
  New value range: [0.0, 1.0]
    Train/validation split applied
    Training: (40000, 32, 32, 3)
    Validation: (10000, 32, 32, 3)
    Test: (10000, 32, 32, 3)


## Exploratory Data Analysis (EDA)

In [4]:
print("\n EXPLORATORY DATA ANALYSIS")

# 1. Sample Images
print("\n  EDA - Generating sample images visualization")
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
fig.suptitle('Sample CIFAR-10 Images (Clean Data)', fontsize=16, fontweight='bold')

for i in range(10):
    ax = axes[i // 5, i % 5]
    ax.imshow(X_train[i])
    ax.set_title(f"{class_names[y_train[i]]}", fontsize=10)
    ax.axis('off')

plt.tight_layout()
plt.savefig('1_sample_images.png', dpi=300, bbox_inches='tight')
print("Saved: 1_sample_images.png")
plt.close()


 EXPLORATORY DATA ANALYSIS

  EDA - Generating sample images visualization
Saved: 1_sample_images.png


In [5]:
# 2. Class Distribution
print("  EDA - Generating class distribution chart")
unique, counts = np.unique(y_train, return_counts=True)

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.bar(class_names, counts, color='steelblue', edgecolor='black', alpha=0.8)
ax.set_xlabel('Class', fontsize=12, fontweight='bold')
ax.set_ylabel('Number of Images', fontsize=12, fontweight='bold')
ax.set_title('CIFAR-10 Training Set Class Distribution', fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')

# Add count labels
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{int(height)}', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('2_class_distribution.png', dpi=300, bbox_inches='tight')
print("Saved: 2_class_distribution.png")
plt.close()

  EDA - Generating class distribution chart
Saved: 2_class_distribution.png


In [6]:
# 3. Pixel Value Distribution
print("  EDA - Generating pixel value distribution")
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Pixel Value Distribution by Channel (Normalized)', fontsize=14, fontweight='bold')

channels = ['Red Channel', 'Green Channel', 'Blue Channel']
colors_hist = ['red', 'green', 'blue']

for i, (channel, color) in enumerate(zip(channels, colors_hist)):
    pixel_values = X_train[:, :, :, i].flatten()
    axes[i].hist(pixel_values, bins=50, color=color, alpha=0.7, edgecolor='black')
    axes[i].set_xlabel('Pixel Value', fontsize=11)
    axes[i].set_ylabel('Frequency', fontsize=11)
    axes[i].set_title(channel, fontsize=12, fontweight='bold')
    axes[i].grid(alpha=0.3, linestyle='--')
    axes[i].set_xlim([0, 1])

plt.tight_layout()
plt.savefig('3_pixel_distribution.png', dpi=300, bbox_inches='tight')
print("Saved: 3_pixel_distribution.png")
plt.close()

  EDA - Generating pixel value distribution
Saved: 3_pixel_distribution.png


In [7]:
# 4. Mean Brightness by Class
print(" EDA - Generating brightness analysis")
brightness_train = X_train.mean(axis=(1, 2, 3))
brightness_by_class = [brightness_train[y_train == i].mean() for i in range(10)]

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.bar(class_names, brightness_by_class, color='coral', edgecolor='black', alpha=0.8)
ax.set_xlabel('Class', fontsize=12, fontweight='bold')
ax.set_ylabel('Mean Brightness (Normalized)', fontsize=12, fontweight='bold')
ax.set_title('Average Image Brightness by Class', fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')

for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.3f}', ha='center', va='bottom', fontsize=9)

plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('4_brightness_analysis.png', dpi=300, bbox_inches='tight')
print("Saved: 4_brightness_analysis.png")
plt.close()

 EDA - Generating brightness analysis
Saved: 4_brightness_analysis.png


## STATISTICS SUMMARY


In [8]:
print("\n DATASET STATISTICS SUMMARY")
print("\n  TRAINING SET STATISTICS:")
print(f"    Mean pixel value: {X_train.mean():.4f}")
print(f"    Std deviation: {X_train.std():.4f}")
print(f"    Min value: {X_train.min():.4f}")
print(f"    Max value: {X_train.max():.4f}")

print("\n  TEST SET STATISTICS:")
print(f"    Mean pixel value: {X_test.mean():.4f}")
print(f"    Std deviation: {X_test.std():.4f}")
print(f"    Min value: {X_test.min():.4f}")
print(f"    Max value: {X_test.max():.4f}")

print("\n  PER-CLASS STATISTICS (Training Set):")
for i, class_name in enumerate(class_names):
    class_images = X_train[y_train == i]
    print(f"    {class_name:12s}: mean={class_images.mean():.4f}, "
          f"std={class_images.std():.4f}, count={len(class_images)}")


 DATASET STATISTICS SUMMARY

  TRAINING SET STATISTICS:
    Mean pixel value: 0.4734
    Std deviation: 0.2516
    Min value: 0.0000
    Max value: 1.0000

  TEST SET STATISTICS:
    Mean pixel value: 0.4766
    Std deviation: 0.2512
    Min value: 0.0000
    Max value: 1.0000

  PER-CLASS STATISTICS (Training Set):
    airplane    : mean=0.5583, std=0.2539, count=5000
    automobile  : mean=0.4576, std=0.2698, count=5000
    bird        : mean=0.4683, std=0.2328, count=5000
    cat         : mean=0.4558, std=0.2578, count=5000
    deer        : mean=0.4383, std=0.2162, count=5000
    dog         : mean=0.4604, std=0.2501, count=5000
    frog        : mean=0.4179, std=0.2289, count=5000
    horse       : mean=0.4662, std=0.2489, count=5000
    ship        : mean=0.5234, std=0.2487, count=5000
    truck       : mean=0.4874, std=0.2729, count=5000


## Data Verification

In [ ]:
plt.savefig('figures/1_sample_images.png', dpi=300, bbox_inches='tight')
plt.savefig('figures/2_class_distribution.png', dpi=300, bbox_inches='tight')
plt.savefig('figures/3_pixel_distribution.png', dpi=300, bbox_inches='tight')
plt.savefig('figures/4_brightness_analysis.png', dpi=300, bbox_inches='tight')

print("\n DATA QUALITY VERIFICATION")
print(f"  Missing values (training): {np.isnan(X_train).sum()}")
print(f"  Missing values (test): {np.isnan(X_test).sum()}")
print(f"  Class balance: {'PERFECT' if len(set(counts)) == 1 else 'IMBALANCED'}")
print(f"  Data type correct: {X_train.dtype == np.float32}")
print(f"  Value range correct: {X_train.min() >= 0 and X_train.max() <= 1}")

print("\nGenerated files:")
print("  - figures/1_sample_images.png")
print("  - figures/2_class_distribution.png")
print("  - figures/3_pixel_distribution.png")
print("  - figures/4_brightness_analysis.png")


 DATA QUALITY VERIFICATION
  Missing values (training): 0
  Missing values (test): 0
  Class balance: PERFECT
  Data type correct: True
  Value range correct: True

Generated files:
  - figures/1_sample_images.png
  - figures/2_class_distribution.png
  - figures/3_pixel_distribution.png
  - figures/4_brightness_analysis.png
